# 01 — EDA: Beijing Multi-Site Air Quality (UCI id=501)

420,768 hourly records — 12 monitoring stations × 35,064 hours, 2013-03-01 to 2017-02-28.

Pollutants: PM2.5, PM10, SO2, NO2, CO, O3.  
Meteorology: TEMP, PRES, DEWP, RAIN, wd (wind direction), WSPM (wind speed).

Goals: coverage and missingness, pollutant distributions, station and seasonal
variation, and the exposure-class imbalance that motivates CTGAN augmentation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(context="notebook", style="whitegrid")
pd.set_option("display.width", 140)

POLLUTANTS = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3"]
METEO = ["TEMP", "PRES", "DEWP", "RAIN", "WSPM"]

In [ ]:
# Written by src/preprocessing/download.py (see README — ucimlrepo cannot serve id=501).
df = pd.read_csv("../data/raw/beijing_multisite.csv")

df["datetime"] = pd.to_datetime(df[["year", "month", "day", "hour"]])
assert len(df) == 420_768, f"expected 420,768 rows, got {len(df):,}"
print(f"{len(df):,} rows × {df.shape[1]} cols | {df.station.nunique()} stations")
print(f"{df.datetime.min()} .. {df.datetime.max()}")
df.head()

## Missingness

Missing values are true `NaN` here — unlike the De Vito dataset, there is **no** `-200`
sentinel to decode. CO is the worst-covered channel.

In [ ]:
miss = (df.isna().mean() * 100).round(2).sort_values(ascending=False)
print(miss[miss > 0])

fig, ax = plt.subplots(figsize=(9, 4))
miss[miss > 0].plot.bar(ax=ax)
ax.set_ylabel("% missing")
ax.set_title("Missingness by channel")
plt.tight_layout()

In [ ]:
# Is missingness clustered in time (sensor outages) or scattered?
gaps = (
    df.set_index("datetime")
      .groupby("station")["PM2.5"]
      .apply(lambda s: s.isna().resample("ME").mean() * 100)
      .unstack(0)
)
fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(gaps.T, cmap="rocket_r", cbar_kws={"label": "% PM2.5 missing"}, ax=ax)
ax.set_title("PM2.5 missingness by station and month")
ax.set_xlabel("")
plt.tight_layout()

## Pollutant distributions

All six are heavy right-tailed — the rare extreme-exposure episodes are exactly the
regime the neckband most needs to get right, and exactly where data is thinnest.

In [ ]:
df[POLLUTANTS].describe(percentiles=[0.5, 0.9, 0.99]).T

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.ravel(), POLLUTANTS):
    sns.histplot(df[col].dropna(), bins=80, ax=ax, log_scale=(False, True))
    ax.set_title(col)
    ax.set_ylabel("count (log)")
fig.suptitle("Pollutant distributions — note the log-scaled counts")
plt.tight_layout()

## Exposure-class imbalance

Binning PM2.5 on the standard AQI breakpoints shows how few samples land in the
categories that trigger an actual health advisory. This is the imbalance the CTGAN
in `src/gan` is meant to correct.

In [ ]:
# PM2.5 (µg/m³) AQI category breakpoints.
BINS = [0, 12, 35.4, 55.4, 150.4, 250.4, np.inf]
LABELS = ["Good", "Moderate", "Unhealthy (sensitive)", "Unhealthy", "Very unhealthy", "Hazardous"]

df["pm25_category"] = pd.cut(df["PM2.5"], bins=BINS, labels=LABELS, right=True)
counts = df["pm25_category"].value_counts().reindex(LABELS)
print(pd.DataFrame({"n": counts, "%": (counts / counts.sum() * 100).round(2)}))

fig, ax = plt.subplots(figsize=(9, 4))
counts.plot.bar(ax=ax, color=sns.color_palette("rocket_r", len(LABELS)))
ax.set_ylabel("hourly records")
ax.set_title("PM2.5 exposure categories — severe class imbalance")
plt.tight_layout()

## Station and seasonal variation

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
order = df.groupby("station")["PM2.5"].median().sort_values().index
sns.boxplot(data=df, x="station", y="PM2.5", order=order, showfliers=False, ax=ax)
ax.set_title("PM2.5 by station")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

In [ ]:
monthly = df.groupby(["month"])[POLLUTANTS].mean()
fig, ax = plt.subplots(figsize=(10, 5))
(monthly / monthly.max()).plot(ax=ax, marker="o")
ax.set_title("Seasonal cycle (each pollutant scaled to its own max)")
ax.set_xlabel("month")
ax.set_xticks(range(1, 13))
plt.tight_layout()

In [ ]:
hourly = df.groupby("hour")[POLLUTANTS].mean()
fig, ax = plt.subplots(figsize=(10, 5))
(hourly / hourly.max()).plot(ax=ax, marker=".")
ax.set_title("Diurnal cycle — relevant to a wearable's exposure timeline")
ax.set_xlabel("hour of day")
plt.tight_layout()

## Correlation structure

Which channels the CTGAN has to reproduce jointly, and which meteorological variables
carry predictive signal. Note `wd` is categorical (16-point compass) and needs encoding
before it enters a model.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
corr = df[POLLUTANTS + METEO].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, square=True, ax=ax)
ax.set_title("Pollutant × meteorology correlation")
plt.tight_layout()

---

### Takeaways for the pipeline

- Missing values are `NaN`, worst on CO (~4.9%); check whether gaps are outage-shaped
  before choosing an imputation strategy.
- Heavy right tails and severe exposure-class imbalance → the case for CTGAN augmentation.
- Strong seasonal and diurnal structure → time-aware splits, not random ones, or the
  evaluation leaks.
- `wd` is categorical; `station` is the grouping key for cross-station generalisation tests.